In [1]:
import pandas as pd 

books = pd.read_csv("books_with_categories.csv")

In [2]:
from transformers import pipeline

classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=None, device="mps")
classifier("I love this!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528684265911579},
  {'label': 'neutral', 'score': 0.005764600355178118},
  {'label': 'anger', 'score': 0.004419781267642975},
  {'label': 'sadness', 'score': 0.002092392183840275},
  {'label': 'disgust', 'score': 0.001611993182450533},
  {'label': 'fear', 'score': 0.0004138521908316761}]]

In [3]:
books["description"][0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world ha

In [4]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}

    for prediction in predictions:   # one sentence's predictions
        for item in prediction:      # each label-score dict
            label = item["label"]
            score = item["score"]
            if label in per_emotion_scores:
                per_emotion_scores[label].append(score)

    return {
        label: np.max(scores) if scores else 0
        for label, scores in per_emotion_scores.items()
    }

In [5]:
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in range(10):
    isbn.append(books["isbn13"].iloc[i])
    sentences = books["description"].iloc[i].split(".")
    sentences = [s.strip() for s in sentences if s.strip()]  # remove empty sentences

    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)

    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [6]:
emotion_scores

{'anger': [np.float64(0.029641257598996162),
  np.float64(0.5944699048995972),
  np.float64(0.04130103439092636),
  np.float64(0.3253040909767151),
  np.float64(0.09173254668712616),
  np.float64(0.20615462958812714),
  np.float64(0.5513834953308105),
  np.float64(0.02365625835955143),
  np.float64(0.30067071318626404),
  np.float64(0.01765839196741581)],
 'disgust': [np.float64(0.3382376730442047),
  np.float64(0.4619906544685364),
  np.float64(0.024568449705839157),
  np.float64(0.12576262652873993),
  np.float64(0.1974342167377472),
  np.float64(0.7573778033256531),
  np.float64(0.1822037398815155),
  np.float64(0.056707724928855896),
  np.float64(0.2874763011932373),
  np.float64(0.17792677879333496)],
 'fear': [np.float64(0.9839729070663452),
  np.float64(0.9352147579193115),
  np.float64(0.9732851982116699),
  np.float64(0.4363398849964142),
  np.float64(0.09504329413175583),
  np.float64(0.036795105785131454),
  np.float64(0.7008540630340576),
  np.float64(0.4044971466064453),
 

In [7]:
from tqdm import tqdm

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"].iloc[i])
    sentences = books["description"].iloc[i].split(".")
    sentences = [s.strip() for s in sentences if s.strip()]  # remove empty sentences

    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)

    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 5197/5197 [04:55<00:00, 17.59it/s]


In [8]:
emotion_df = pd.DataFrame(emotion_scores)
emotion_df["isbn13"] = isbn

emotion_df.head()

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.029641,0.338238,0.983973,0.949027,0.956065,0.729603,0.697846,9780002005883
1,0.594470,0.461991,0.935215,0.704421,0.051414,0.212222,0.891110,9780002261982
2,0.041301,0.024568,0.973285,0.767237,0.010860,0.009796,0.042176,9780006178736
3,0.325304,0.125763,0.436340,0.242209,0.043272,0.029084,0.732686,9780006280897
4,0.091733,0.197434,0.095043,0.041146,0.475881,0.074878,0.890048,9780006280934


In [9]:
books = pd.merge(books, emotion_df, on = "isbn13")

In [10]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,...,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,...,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction,0.029641,0.338238,0.983973,0.949027,0.956065,0.729603,0.697846
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction,0.594470,0.461991,0.935215,0.704421,0.051414,0.212222,0.891110
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,...,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction,0.041301,0.024568,0.973285,0.767237,0.010860,0.009796,0.042176
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,...,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Non-Fiction,0.325304,0.125763,0.436340,0.242209,0.043272,0.029084,0.732686
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,...,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Non-Fiction,0.091733,0.197434,0.095043,0.041146,0.475881,0.074878,0.890048
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,...,Mistaken Identity,9788172235222 On A Train Journey Home To North...,Non-Fiction,0.204143,0.053939,0.923713,0.303277,0.974265,0.028639,0.807058
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,...,Journey to the East,9788173031014 This book tells the tale of a ma...,Non-Fiction,0.058449,0.127941,0.025688,0.400263,0.016014,0.227765,0.891073
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,...,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...,Fiction,0.011246,0.010868,0.314812,0.942169,0.060436,0.056820,0.344738
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,...,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...,Non-Fiction,0.034505,0.080505,0.409872,0.776052,0.317456,0.049227,0.950763


In [12]:
books.to_csv("books_with_emotions.csv", index=False)